# 01 · Variables predictivas, etiqueta objetivo y conjuntos temporales

Este notebook transforma las particiones mensuales de `estacion_hora` en una base preparada para modelado. Se trabaja por meses para no cargar los más de ocho millones de filas en memoria.

La predicción se formula para **una hora vista**: a partir de la información conocida en `t`, se estima el riesgo de vaciado o saturación en `t + 1 hora`.

## Criterios metodológicos

- Solo se conservan estaciones activas con capacidad positiva.
- Riesgo de vaciado: ocupación futura menor o igual al 10 %.
- Riesgo de saturación: ocupación futura mayor o igual al 90 %.
- Las variables de flujo se retrasan una hora para evitar usar información que todavía no se conocería al hacer la predicción.
- El año 2020 queda identificado como `eda_excluded` y no se usa para entrenar el modelo principal.
- Entrenamiento: 2019, 2021 y enero-septiembre de 2022; validación: octubre de 2022; test final: noviembre-diciembre de 2022.

In [ ]:
# Importamos únicamente las librerías necesarias para transformar CSV grandes.
from pathlib import Path
import csv

import numpy as np
import pandas as pd

# Localizamos el proyecto a partir de la carpeta en la que se guarda este notebook.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Datos analiticos').exists():
    # Esta alternativa funciona si Jupyter se abre desde la carpeta notebooks.
    PROJECT_ROOT = PROJECT_ROOT.parent

INPUT_DIR = PROJECT_ROOT / 'Datos analiticos' / 'estacion_hora'
OUTPUT_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
MANIFEST_PATH = OUTPUT_DIR / 'manifest_features.csv'

# Definimos explícitamente la regla de negocio de riesgo para facilitar su ajuste posterior.
RISK_EMPTY_THRESHOLD = 0.10
RISK_FULL_THRESHOLD = 0.90
PREDICTION_HORIZON = pd.Timedelta(hours=1)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
state_files = sorted(INPUT_DIR.glob('estacion_hora_*.csv'))

assert state_files, f'No se han encontrado CSV en {INPUT_DIR}'
print(f'Particiones de entrada: {len(state_files)}')
print(f'Carpeta de salida: {OUTPUT_DIR}')

In [ ]:
# Estas funciones pequeñas evitan duplicar lógica y hacen explícitos los controles temporales.
def read_station_hour(path: Path) -> pd.DataFrame:
    """Lee una partición y conserva station_id como texto para no alterar identificadores."""
    # low_memory=False evita inferencias inconsistentes en columnas textuales con valores vacíos.
    frame = pd.read_csv(path, dtype={'station_id': 'string', 'station_number': 'string'}, low_memory=False)
    frame['fecha_hora_dt'] = pd.to_datetime(frame['fecha_hora_local'], errors='coerce')
    return frame


def exact_lag(frame: pd.DataFrame, column: str, hours: int) -> pd.Series:
    """Devuelve el valor de una estación hace N horas solo si no existe un hueco temporal."""
    grouped = frame.groupby('station_id', group_keys=False)
    lagged_value = grouped[column].shift(hours)
    lagged_time = grouped['fecha_hora_dt'].shift(hours)
    expected_time = frame['fecha_hora_dt'] - pd.Timedelta(hours=hours)
    return lagged_value.where(lagged_time.eq(expected_time))


def exact_rolling_mean(frame: pd.DataFrame, column: str, window_hours: int) -> pd.Series:
    """Calcula la media de las horas anteriores, sin usar la hora actual ni atravesar huecos."""
    grouped = frame.groupby('station_id', group_keys=False)
    # shift(1) impide que el valor de la hora t entre en una variable usada para predecir t.
    rolling_mean = grouped[column].transform(
        lambda series: series.shift(1).rolling(window_hours, min_periods=window_hours).mean()
    )
    oldest_time = grouped['fecha_hora_dt'].shift(window_hours)
    expected_oldest = frame['fecha_hora_dt'] - pd.Timedelta(hours=window_hours)
    return rolling_mean.where(oldest_time.eq(expected_oldest))


def assign_split(timestamp: pd.Series) -> pd.Series:
    """Asigna train, validation, test o eda_excluded respetando el orden temporal."""
    split = pd.Series('not_used', index=timestamp.index, dtype='string')
    split.loc[timestamp.dt.year.eq(2020)] = 'eda_excluded'
    split.loc[(timestamp < pd.Timestamp('2022-10-01')) & ~timestamp.dt.year.eq(2020)] = 'train'
    split.loc[(timestamp >= pd.Timestamp('2022-10-01')) & (timestamp < pd.Timestamp('2022-11-01'))] = 'validation'
    split.loc[(timestamp >= pd.Timestamp('2022-11-01')) & (timestamp < pd.Timestamp('2023-01-01'))] = 'test'
    return split


In [ ]:
def build_feature_partition(position: int) -> dict:
    """Genera las variables de un mes y devuelve métricas de control de calidad."""
    source_path = state_files[position]
    period = source_path.stem.rsplit('_', 1)[1]
    destination = OUTPUT_DIR / f'estacion_hora_features_{period}.csv'

    current = read_station_hour(source_path)

    # Si el mes ya se generó correctamente, se conserva y se resumen sus etiquetas para reconstruir el manifiesto.
    if destination.exists():
        existing = pd.read_csv(destination, usecols=['risk_class_1h', 'risk_empty_1h', 'risk_full_1h'], low_memory=False)
        return {
            'period': period,
            'status': 'already_exists',
            'input_rows': len(current),
            'output_rows': len(existing),
            'rows_with_target': int(existing['risk_class_1h'].notna().sum()),
            'risk_empty_rows': int(existing['risk_empty_1h'].eq(1).sum()),
            'risk_full_rows': int(existing['risk_full_1h'].eq(1).sum()),
        }

    current['is_current_period'] = True

    # Añadimos las últimas 24 horas del mes anterior para calcular retardos y medias móviles.
    context_frames = [current]
    if position > 0:
        previous = read_station_hour(state_files[position - 1])
        cutoff = previous['fecha_hora_dt'].max() - pd.Timedelta(hours=24)
        previous = previous.loc[previous['fecha_hora_dt'] >= cutoff].copy()
        previous['is_current_period'] = False
        context_frames.insert(0, previous)

    # Añadimos la primera hora del mes siguiente para etiquetar correctamente la última hora actual.
    if position < len(state_files) - 1:
        next_month = read_station_hour(state_files[position + 1])
        first_hour = next_month['fecha_hora_dt'].min()
        next_month = next_month.loc[next_month['fecha_hora_dt'].eq(first_hour)].copy()
        next_month['is_current_period'] = False
        context_frames.append(next_month)

    work = pd.concat(context_frames, ignore_index=True)
    work = work.sort_values(['station_id', 'fecha_hora_dt']).reset_index(drop=True)

    # Aplicamos el filtro solicitado: estación activa y capacidad utilizable.
    work['activation'] = pd.to_numeric(work['activation'], errors='coerce')
    work['capacity'] = pd.to_numeric(work['capacity'], errors='coerce')
    work['bikes_available'] = pd.to_numeric(work['bikes_available'], errors='coerce')
    work['net_flow'] = pd.to_numeric(work['net_flow'], errors='coerce')
    work['departures_count'] = pd.to_numeric(work['departures_count'], errors='coerce')
    work['arrivals_count'] = pd.to_numeric(work['arrivals_count'], errors='coerce')
    work['occupancy_ratio'] = pd.to_numeric(work['occupancy_ratio'], errors='coerce')
    work['valid_station_row'] = work['activation'].eq(1) & work['capacity'].gt(0)

    # Construimos la etiqueta futura y comprobamos que la siguiente observación sea exactamente t + 1 hora.
    grouped = work.groupby('station_id', group_keys=False)
    future_time = grouped['fecha_hora_dt'].shift(-1)
    future_occupancy = grouped['occupancy_ratio'].shift(-1)
    future_valid_station = grouped['valid_station_row'].shift(-1)
    # Exigimos también que la estación siga activa y con capacidad válida en t + 1.
    valid_future = future_time.eq(work['fecha_hora_dt'] + PREDICTION_HORIZON) & future_occupancy.notna() & future_valid_station.eq(True)

    work['future_occupancy_ratio_1h'] = future_occupancy.where(valid_future)
    work['risk_empty_1h'] = pd.Series(pd.NA, index=work.index, dtype='Int8')
    work['risk_full_1h'] = pd.Series(pd.NA, index=work.index, dtype='Int8')
    work.loc[valid_future, 'risk_empty_1h'] = (future_occupancy.loc[valid_future] <= RISK_EMPTY_THRESHOLD).astype('int8')
    work.loc[valid_future, 'risk_full_1h'] = (future_occupancy.loc[valid_future] >= RISK_FULL_THRESHOLD).astype('int8')

    # La clase resume el objetivo multiclase: 0 estable, 1 riesgo de vaciado, 2 riesgo de saturación.
    work['risk_class_1h'] = pd.Series(pd.NA, index=work.index, dtype='Int8')
    work.loc[valid_future, 'risk_class_1h'] = 0
    work.loc[valid_future & work['risk_empty_1h'].eq(1), 'risk_class_1h'] = 1
    work.loc[valid_future & work['risk_full_1h'].eq(1), 'risk_class_1h'] = 2

    # Extraemos variables temporales directamente de la marca de tiempo de la observación.
    work['hour'] = work['fecha_hora_dt'].dt.hour
    work['day_of_week'] = work['fecha_hora_dt'].dt.dayofweek
    work['month'] = work['fecha_hora_dt'].dt.month
    work['week_of_year'] = work['fecha_hora_dt'].dt.isocalendar().week.astype('int16')

    # Generamos retardos de disponibilidad y de actividad; todos proceden de horas anteriores.
    for column in ['occupancy_ratio', 'bikes_available', 'net_flow', 'departures_count', 'arrivals_count']:
        for lag in [1, 2, 24]:
            work[f'{column}_lag_{lag}h'] = exact_lag(work, column, lag)

    # Las medias móviles también se calculan con información anterior a la hora que se predice.
    for column in ['occupancy_ratio', 'net_flow', 'departures_count', 'arrivals_count']:
        for window in [3, 24]:
            work[f'{column}_mean_previous_{window}h'] = exact_rolling_mean(work, column, window)

    # Marcamos la partición temporal antes de retirar el contexto usado exclusivamente para calcular variables.
    work['dataset_split'] = assign_split(work['fecha_hora_dt'])
    result = work.loc[work['is_current_period'] & work['valid_station_row']].copy()
    result = result.drop(columns=['is_current_period', 'valid_station_row', 'fecha_hora_dt'])

    # Se escribe primero un temporal y se publica el CSV solo cuando el mes termina correctamente.
    temporary = destination.with_suffix('.csv.part')
    result.to_csv(temporary, index=False, encoding='utf-8-sig')
    temporary.replace(destination)

    return {
        'period': period,
        'status': 'created',
        'input_rows': len(current),
        'output_rows': len(result),
        'rows_with_target': int(result['risk_class_1h'].notna().sum()),
        'risk_empty_rows': int(result['risk_empty_1h'].eq(1).sum()),
        'risk_full_rows': int(result['risk_full_1h'].eq(1).sum()),
    }


In [ ]:
# Ejecutamos la transformación mes a mes y almacenamos un resumen auditable.
metrics = []
for position in range(len(state_files)):
    metric = build_feature_partition(position)
    metrics.append(metric)
    print(metric)

manifest = pd.DataFrame(metrics)
manifest.to_csv(MANIFEST_PATH, index=False, encoding='utf-8-sig')
manifest

In [ ]:
# Verificamos una muestra de columnas y el reparto temporal antes de entrenar modelos.
feature_files = sorted(OUTPUT_DIR.glob('estacion_hora_features_*.csv'))
assert len(feature_files) == len(state_files), 'Faltan particiones de variables'

sample = pd.read_csv(feature_files[-1], nrows=5)
required_columns = {
    'risk_empty_1h', 'risk_full_1h', 'risk_class_1h', 'dataset_split',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_mean_previous_24h',
    'hour', 'day_of_week', 'month'
}
assert required_columns.issubset(sample.columns), 'Faltan variables requeridas'

print(f'Particiones generadas: {len(feature_files)}')
print('Filas por conjunto temporal:')
display(manifest.groupby('status', dropna=False).size())
sample[list(required_columns)].head()